# 第 1 周 · 周六：异动分析入门——指标变了，然后呢？

> 今日目标：掌握异动分析的三步法，能在30分钟内从"指标异常"定位到"根因+建议"
> 配对课程：01python_learn W1D6（文件读写与CSV分析器）
> 扩展阅读：02analyst_thinking/week_05/day_04（异动归因）

---

## 一、核心概念与第一性原理

### 1.1 什么是异动分析

异动分析 = 当指标出现异常变化时，系统性地从"指标变了"定位到"什么变了、为什么变、该怎么办"。

每天都有指标在变——流量涨了、转化率跌了、客单价稳如老狗。大多数波动是正常的随机噪音，但有些变化意味着"出事了"。分析师的工作就是**从噪音中识别信号**。

### 1.2 从第一性原理推导：为什么需要"结构化"的异动分析

当你看到"DAU跌了15%"时，你的大脑会同时冒出无数可能性：是数据bug？是新用户少了？是老用户不来了？是某个渠道挂了？是服务端出问题了？

如果你的排查方式是"一个一个试"——你会在50个方向中浪费大量时间。

**结构化的异动分析 = 用一套固定的"排查顺序"，每一步排除掉最大的一批可能性。**

### 1.3 异动分析三步法

```
第一步：定位（Locate）—— 哪个指标变了？变多大？什么时候开始的？是真实变化还是数据bug？
    ↓
第二步：拆解（Decompose）—— 变化来自哪个因子（乘）？哪个维度（加）？
    ↓
第三步：解释（Explain）—— 为什么这个因子/维度变了？（内部改动/外部事件/数据问题）
```

每一步都在缩小的搜索范围。三步之后，你不再是"面对一个模糊的问题"，而是"面对一个具体的原因"。

> **今天最重要的认知**：异动分析的价值不是"找到答案"（答案往往很简单），而是"快速缩小搜索范围"。一个能在5分钟内把"DAU为什么跌"缩小到"Web端登录态问题"的分析师，和一个花2小时拉了50张表才得出同样结论的分析师——工作效率差24倍。

---

## 二、知识点详细拆解

### 2.1 第一步：定位——判断"是不是真的异常"

**为什么重要**：很多"异常"其实是数据bug、统计口径变化、或正常的周期性波动。

定位三问：
1. **有没有数据问题？** 埋点挂了？ETL延迟了？统计口径变了？
2. **是不是正常波动？** 过去90天的日波动率通常在什么范围？今天的跌幅是否超出2倍标准差？
3. **什么时候开始的？** 是某一天的突变（→大概率跟当天的事件有关）还是持续趋势（→可能跟产品/策略变化有关）？

### 2.2 第二步：拆解——找到"出问题的环节"

**为什么重要**：一个总体指标的变化，一定会在某个子指标上体现。

两个层次的拆解（复用W1D5的方法）：

| 层次 | 方法 | 回答的问题 | 举例 |
|------|------|-----------|------|
| 因子拆解 | 乘法拆解 | "是哪个环节出了问题？" | DAU跌了→是新/留/回哪个跌了？ |
| 维度拆解 | 加法拆解 | "是哪个群体/品类/渠道？" | 留存用户跌了→是哪个平台/版本的留存跌了？ |

先做因子拆解（锁定环节），再做维度拆解（锁定群体）。

### 2.3 第三步：解释——从"什么变了"到"为什么变"

**为什么重要**：知道了"什么变了"还不够——你需要知道"为什么会变"才能给出建议。

归因三问：
1. **内部有什么变动？** 上线了新功能？改了算法？活动到期了？运营做了什么操作？
2. **外部有什么事件？** 节假日？竞对做了大促？热搜事件？天气？
3. **数据本身有问题吗？** 埋点改了？统计口径变了？

**关键：分析师的50%价值在于知道"这个异常应该问哪个团队"**——不用自己查所有可能，直接问最可能知道答案的人。

### 2.4 异动分析报告的标准格式

一个好的异动分析结果应该能用三句话说清楚：
```
第一句（发生了什么）：XX指标在X月X日跌了X%，超出正常波动范围（平时±5%，今天-27%）。
第二句（为什么会发生）：拆解发现主要由XX品类的XX环节变化导致，根因是XX活动到期。
第三句（该怎么办）：建议恢复活动/启动备选方案。预计恢复后指标可回到正常水平。
```

---

## 三、动手练习

### 练习1：模拟渠道层面的异动分析（基础）

In [1]:
# ===== 练习1：次留异动分析 =====

channels_retention = {
    "渠道":    ["应用商店", "信息流", "社媒", "SEO", "其他"],
    "上周新用户占比": [0.30, 0.35, 0.15, 0.12, 0.08],
    "上周次留":     [0.28, 0.24, 0.30, 0.22, 0.20],
    "本周新用户占比": [0.25, 0.45, 0.12, 0.10, 0.08],
    "本周次留":     [0.28, 0.12, 0.30, 0.22, 0.20],
}

print("新用户次日留存率异动分析")
print("=" * 60)

last_week_total = sum(a*b for a,b in zip(channels_retention["上周新用户占比"], channels_retention["上周次留"]))
this_week_total = sum(a*b for a,b in zip(channels_retention["本周新用户占比"], channels_retention["本周次留"]))

print(f"上周整体次留: {last_week_total:.1%}")
print(f"本周整体次留: {this_week_total:.1%}")
print(f"变化: {(this_week_total - last_week_total)*100:.1f}个百分点")
print()

print(f"{'渠道':<10} {'上周占比':>8} {'上周次留':>8} {'本周占比':>8} {'本周次留':>8} {'次留变化':>8}")
print("-" * 60)
for i in range(len(channels_retention["渠道"])):
    ch = channels_retention["渠道"][i]
    lp, lr = channels_retention["上周新用户占比"][i], channels_retention["上周次留"][i]
    tp, tr = channels_retention["本周新用户占比"][i], channels_retention["本周次留"][i]
    print(f"{ch:<10} {lp:>8.0%} {lr:>8.0%} {tp:>8.0%} {tr:>8.0%} {(tr-lr)*100:>+7.1f}pp")

print()
print("三步法结论:")
print("  定位: 整体次留从25%跌到18%，属于异常（跌幅>5pp）")
print("  拆解: 信息流次留从24%暴跌到12%(-50%)，同时其新用户占比从35%涨到45%")
print("  解释: 大概率是信息流投放素材或定向出了问题→吸引了大量非目标用户")
print("  建议: 暂停信息流投放→检查素材和定向→修复后小预算重启")
print()
print("【预期输出】定位到信息流渠道，给出明确的根因假设和行动建议")

新用户次日留存率异动分析
上周整体次留: 25.5%
本周整体次留: 19.8%
变化: -5.7个百分点

渠道             上周占比     上周次留     本周占比     本周次留     次留变化
------------------------------------------------------------
应用商店            30%      28%      25%      28%    +0.0pp
信息流             35%      24%      45%      12%   -12.0pp
社媒              15%      30%      12%      30%    +0.0pp
SEO             12%      22%      10%      22%    +0.0pp
其他               8%      20%       8%      20%    +0.0pp

三步法结论:
  定位: 整体次留从25%跌到18%，属于异常（跌幅>5pp）
  拆解: 信息流次留从24%暴跌到12%(-50%)，同时其新用户占比从35%涨到45%
  解释: 大概率是信息流投放素材或定向出了问题→吸引了大量非目标用户
  建议: 暂停信息流投放→检查素材和定向→修复后小预算重启

【预期输出】定位到信息流渠道，给出明确的根因假设和行动建议


### 练习2：用CSV数据做异动分析（进阶，配对01 W1D6）

In [2]:
# ===== 练习2：CSV文件读写 + 异动分析 =====

data_content = """date,channel,new_users,retention_1d
2026-01-01,organic,1200,0.28
2026-01-01,paid,800,0.22
2026-01-01,social,500,0.32
2026-01-02,organic,1150,0.27
2026-01-02,paid,900,0.21
2026-01-02,social,480,0.31
2026-01-03,organic,1100,0.29
2026-01-03,paid,1500,0.10
2026-01-03,social,450,0.30
"""

with open("retention_data.csv", "w") as f:
    f.write(data_content)

print("已创建 retention_data.csv")
print()

# 读取并手动分析
with open("retention_data.csv", "r") as f:
    lines = f.readlines()

print("原始数据:")
for line in lines:
    print("  " + line.strip())

print()
print("异动分析:")
print("  观察1: 1月3日paid渠道新用户从800-900骤增到1500(+67%)")
print("  观察2: 但次留从21-22%暴跌到10%(-55%)")
print("  结论: 典型的'花钱买量但质量暴跌'——投放素材/定向出问题了")
print()
print("【预期输出】识别出1月3日paid渠道的异常并给出根因假设")

已创建 retention_data.csv

原始数据:
  date,channel,new_users,retention_1d
  2026-01-01,organic,1200,0.28
  2026-01-01,paid,800,0.22
  2026-01-01,social,500,0.32
  2026-01-02,organic,1150,0.27
  2026-01-02,paid,900,0.21
  2026-01-02,social,480,0.31
  2026-01-03,organic,1100,0.29
  2026-01-03,paid,1500,0.10
  2026-01-03,social,450,0.30

异动分析:
  观察1: 1月3日paid渠道新用户从800-900骤增到1500(+67%)
  观察2: 但次留从21-22%暴跌到10%(-55%)
  结论: 典型的'花钱买量但质量暴跌'——投放素材/定向出问题了

【预期输出】识别出1月3日paid渠道的异常并给出根因假设


### 练习3：模拟"5分钟救火"——从收到告警到给出初步结论（挑战）

In [3]:
# ===== 练习3：模拟救火场景 =====

print("=" * 50)
print("练习3：5分钟救火——DAU突然跌了15%")
print("=" * 50)
print()
print("场景: 周五下午4:55, 运营VP在群里@你")
print("      '今天DAU怎么比昨天少了15%？！'")
print()
print("你的5分钟应急:")
print()
print("T+0min [确认数据]:")
print("  - 昨天DAU=102万, 今天=87万, 确实少了15万")
print("  - 检查ETL: 数据已完整入库(排除数据延迟)")
print()
print("T+1min [PDS拆解]:")
print("  新增: 12万(平时11万，正常)")
print("  留存: 55万(平时70万，少15万！) ← 问题在这")
print("  回流: 20万(平时21万，正常)")
print()
print("T+2min [维度下钻]:")
print("  iOS留存: 正常")
print("  Android留存: 正常")
print("  Web端留存: 从8万跌到3万(-63%) ← 找到了！")
print()
print("T+3min [问Web团队]:")
print("  '今天上线了什么？'→'上午发版，登录态丢了'")
print()
print("T+5min [回复VP]:")
print("  'Web端发版导致登录态丢失，约5万老用户被当做未登录用户。'")
print("  'Web团队正在修复，预计1小时内恢复。'")
print()
print("关键能力:")
print("  1. 不慌——先确认是不是真异常(数据质量)")
print("  2. 快速拆解——PDS+维度下钻, 5分钟缩小范围")
print("  3. 知道问谁——不用自己查, 直接找最可能知道的人")
print()
print("【预期输出】完整的5分钟分析链路，从定位到解释到回复")

练习3：5分钟救火——DAU突然跌了15%

场景: 周五下午4:55, 运营VP在群里@你
      '今天DAU怎么比昨天少了15%？！'

你的5分钟应急:

T+0min [确认数据]:
  - 昨天DAU=102万, 今天=87万, 确实少了15万
  - 检查ETL: 数据已完整入库(排除数据延迟)

T+1min [PDS拆解]:
  新增: 12万(平时11万，正常)
  留存: 55万(平时70万，少15万！) ← 问题在这
  回流: 20万(平时21万，正常)

T+2min [维度下钻]:
  iOS留存: 正常
  Android留存: 正常
  Web端留存: 从8万跌到3万(-63%) ← 找到了！

T+3min [问Web团队]:
  '今天上线了什么？'→'上午发版，登录态丢了'

T+5min [回复VP]:
  'Web端发版导致登录态丢失，约5万老用户被当做未登录用户。'
  'Web团队正在修复，预计1小时内恢复。'

关键能力:
  1. 不慌——先确认是不是真异常(数据质量)
  2. 快速拆解——PDS+维度下钻, 5分钟缩小范围
  3. 知道问谁——不用自己查, 直接找最可能知道的人

【预期输出】完整的5分钟分析链路，从定位到解释到回复


---

## 四、业务场景应用

### 场景1：下单转化率突然跌了27%——一个完整的异动分析案例

早上10点，运营总监在群里发了一张图：昨天的下单转化率从平时的5.2%跌到了3.8%，跌了27%。

**定位**：确认数据无误（埋点上报量正常）。确认跌幅异常（过去90天从未低于4.5%）。

**拆解-因子**：下单转化率 = 商品页→下单页 × 下单页→支付成功。拆开：商品页→下单页从12%→8%(-33%)，下单页→支付43%不变。问题在"用户看到商品页后为什么不下单了"。

**拆解-维度**：按品类再拆——服装从13%→7%，数码从11%→12%。问题集中在服装品类。

**解释**：检查服装品类——发现昨天下午运营把服装"满200减50"活动券到期下架，忘记续期。用户看到价格涨回去了，不下单了。

**行动**：立即恢复优惠券。

**用vs不用**：不用三步法→花2小时拉了20张表才发现问题；用三步法→15分钟定位根因。

### 场景2："最近好像生意变差了"——从感觉出发的异动分析

老板说"最近生意好像变差了"。你不确定ta的感觉对不对——所以第一步是：

1. 先拉最近90天的日GMV趋势——发现最近2周确实在持续下降（不是"某一天突然跌"）
2. 拆解GMV——发现客单价稳定，但订单数在降
3. 拆解订单数——发现转化率在降，但流量在涨
4. 这说明：买量在加大但转化在变差——用户质量在恶化
5. 维度拆解——发现主要是新用户转化率在降（老用户转化稳定）
6. 根因：最近2周投放团队换了定向策略，拉来的人跟产品不匹配

**用vs不用**：不用异动分析→"好像是有点差，多投点广告吧"（问题更严重）；用异动分析→定位到投放定向问题，针对性调整。

### 场景3：Session时长突然翻倍——指标暴涨也可能是坏事

用户平均Session时长从8分钟涨到22分钟。直觉觉得"用户更沉浸了"。

但用异动分析：
- 拆解Session内的行为数——正常Session有8-12个行为事件，现在80%的Session只有1个事件
- 看时间分布——异常从昨天下午2点开始，刚好是后端发版时间
- 结论：不是用户更沉浸了，是埋点bug——心跳上报失败了，Session一直等到30分钟超时才关闭

**用vs不用**：不用异动分析→"Session时长翻倍，用户体验在变好！"；用异动分析→"埋点bug，赶紧修"。

---

## 五、分级检验清单

### 初级检验
- [ ] 能说出异动分析的三步法（定位→拆解→解释）
- [ ] 当指标变化时，能区分"正常波动"和"需要追查的异常"
- [ ] 能用加法拆解定位到出问题的维度

**答案提示**：正常波动=在历史波动范围内(±2σ)；异常=超出范围或趋势性变化。

### 中级检验
- [ ] 能结合乘法拆解和加法拆解进行两层下钻（先因子，后维度）
- [ ] 能在30分钟内完成一个中等复杂度的异动分析并写出"三句话"结论
- [ ] 能识别"指标涨了但不是好事"的场景（如Session时长涨了但可能是bug）

**答案提示**：两层下钻=先乘后加；三句话=发生了什么+为什么+怎么办。

### 高级检验
- [ ] 能设计异动监控的预警阈值（什么程度才告警？告警频率如何避免疲劳？）
- [ ] 能在不确定中找到最可能的方向（基于经验判断优先级），而不需要一个一个排查
- [ ] 能在压力下（VP在群里催）有条理地完成分析，5分钟给出方向性结论

**答案提示**：预警阈值=基于历史波动率（如日环比>10%告警）；优先级=从最常见的原因开始排查（新增渠道/服务变更/活动到期）。

---

## 六、跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 本周全部** | 指标/OSM/CMOP/拆解 | 前五天的每一个技能都是今天异动分析的基础 |
| **→ 明日** | 周复习 | 明天用本周全套工具做一个完整的分析方案 |
| **→ W2** | AARRR/漏斗 | 增长分析中每天都在做异动分析 |
| **→ W7** | AB实验/统计推断 | 学了统计后可以用控制图+假设检验判断"异常" |
| **扩展** | 02analyst_thinking W5D4 | 异动归因的深入方法论 |

---

## 七、当日小结

> **今天最重要的一个认知**：异动分析的价值不是"找到答案"（答案往往很简单），而是"快速缩小搜索范围"。三步法（定位→拆解→解释）是你的加速器——每一步都在排除掉一大批不相关的可能性，让正确答案快速浮出水面。

**三条金句带回家**：
1. 看到指标异常，先判断"是不是真的异常"——80%的"告警"其实是数据bug或正常波动
2. 先乘后加——乘法找环节，加法找群体
3. 知道问谁比会查数据更重要——分析师的50%价值在于知道"这个问题应该问哪个团队"

### 课后拓展
1. **练习**：找一个你日常工作中最近发生过的指标变化，用三步法重新分析一遍。你在哪一步花了最长时间？下次怎么做能更快？
2. **思考**：如果三个因子同时变了（流量-5%，转化率+8%，客单价-3%），你怎么判断哪个是"主要矛盾"？
3. **预习**：明天的周复习会用一个完整案例（在线教育平台），让你综合运用本周的所有框架。

---

*03analyst_thinking · Week 1 Day 6 · 2026-05-21*